# agent101 — Build Mini Claude Code from Zero

A step-by-step tutorial that teaches you how to build a mini AI coding agent from scratch.

Based on [learn-claude-code](https://github.com/shareAI-lab/learn-claude-code) by shareAI-lab.

---

# S01: The Agent Loop

`[ s01 ] s02 > s03 > s04 > s05 > s06 | s07 > s08 > s09 > s10 > s11 > s12`

> *"One loop & PowerShell is all you need"* — one tool + one loop = an agent.
>
> **Harness layer**: The loop — the model’s first connection to the real world.

## Problem

A language model can reason about code, but it can’t *touch* the real world — can’t read files, run tests, or check errors. Without a loop, every tool call requires you to manually copy-paste results back. **You become the loop.**

## Solution

```
+--------+      +-------+      +---------+
|  User  | ---> |  LLM  | ---> |  Tool   |
| prompt |      |       |      | execute |
+--------+      +---+---+      +----+----+
                    ^                |
                    |   tool_result  |
                    +----------------+
                    (loop until model stops calling tools)
```

One exit condition controls the entire flow. The loop runs until the model stops calling tools.

### Step 1: Send a message to the LLM with a tool

First, let’s set up the client, define a `powershell` tool, and send a simple message to see how the LLM responds when it has a tool available.

In [1]:
import os
import json
import subprocess
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
)
MODEL = os.getenv("AZURE_OPENAI_DEPLOYMENT")

SYSTEM = f"You are a coding agent at {os.getcwd()}. You MUST use the powershell tool to run commands. Never output raw JSON — always call tools via function calling. Be direct, no explanations."

TOOLS = [{
    "type": "function",
    "function": {
        "name": "powershell",
        "description": "Run a PowerShell command.",
        "parameters": {
            "type": "object",
            "properties": {"command": {"type": "string"}},
            "required": ["command"],
        },
    },
}]

USER_MSG = "show current directory"
# Send a test message and inspect the raw response
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": USER_MSG},
    ],
    tools=TOOLS,
)

msg = response.choices[0].message
print("=== Raw Response ===")
print(f"content:    {msg.content}")
print(f"tool_calls: {msg.tool_calls}")
print()
if msg.tool_calls:
    tc = msg.tool_calls[0]
    print("=== First Tool Call ===")
    print(f"id:        {tc.id}")
    print(f"function:  {tc.function.name}")
    print(f"arguments: {tc.function.arguments}")

=== Raw Response ===
content:    None
tool_calls: [ChatCompletionMessageFunctionToolCall(id='call_cy4EEGG5CDwtZM4cvNMs5In8', function=Function(arguments='{"command":"Get-Location"}', name='powershell'), type='function')]

=== First Tool Call ===
id:        call_cy4EEGG5CDwtZM4cvNMs5In8
function:  powershell
arguments: {"command":"Get-Location"}


**What happened?** The LLM didn’t reply with text — it replied with a **tool call**. It wants us to run a PowerShell command.

But nobody executed it! The model can’t touch the real world on its own. That’s what the agent loop is for — we need code that:
1. Executes the tool call
2. Sends the result back to the LLM
3. Repeats until the model stops asking for tools

### Step 2: The Tool Handler

A simple function that executes a PowerShell command and returns the output. Includes basic safety checks and a timeout.



In [2]:
def run_powershell(command: str) -> str:
    """Execute a PowerShell command with safety guards."""
    dangerous = ["Remove-Item -Recurse -Force /", "shutdown", "Restart-Computer", "Stop-Computer"]
    if any(d in command for d in dangerous):
        return "Error: Dangerous command blocked"
    try:
        r = subprocess.run(
            ["powershell", "-NoProfile", "-Command", command],
            cwd=os.getcwd(), capture_output=True, text=True, timeout=120,
        )
        out = (r.stdout + r.stderr).strip()
        return out[:50000] if out else "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Timeout (120s)"
    except (FileNotFoundError, OSError) as e:
        return f"Error: {e}"

# Quick test
print(run_powershell("Write-Output 'Hello from the agent!'"))

Hello from the agent!


### make real tool call
1. **User prompt** becomes the first message
2. **Send messages + tool definitions** to the LLM
3. **Check the response** — if the model didn’t call a tool, we’re done
4. **Execute each tool call**, collect results, append as messages, loop back to step 2

### Step 3: The Agent Loop


Let’s build it piece by piece.

This is the **entire secret** of an AI coding agent — a `while True` loop that:
1. Calls the LLM with the conversation history + tools
2. Appends the assistant’s response
3. If the model didn’t call any tools → **done** (exit)
4. Otherwise, execute each tool call, append results, and **loop back**

In [3]:
TOOL_HANDLERS = {"powershell": run_powershell}

def agent_loop(messages: list):
    """Core agent loop: call LLM, execute tools, repeat until done."""
    turn = 0
    while True:
        turn += 1
        all_messages = [{"role": "system", "content": SYSTEM}] + messages

        # Show what we send to the LLM
        print(f"\n{'='*60}")
        print(f"Turn {turn} - LLM Input ({len(all_messages)} messages):")
        print(f"{'='*60}")
        for j, m in enumerate(all_messages):
            role = m["role"]
            if role == "system":
                print(f"  [{j}] system: {m['content'][:80]}...")
            elif role == "user":
                content = m.get("content", "")
                if isinstance(content, str):
                    print(f"  [{j}] user: {content[:100]}")
                else:
                    print(f"  [{j}] user: (tool results)")
            elif role == "assistant":
                tc = m.get("tool_calls")
                if tc:
                    print(f"  [{j}] assistant: (called {len(tc)} tool(s))")
                else:
                    print(f"  [{j}] assistant: {str(m.get('content', ''))[:100]}")
            elif role == "tool":
                print(f"  [{j}] tool: {m['content'][:80]}")

        # 1. Call the LLM
        response = client.chat.completions.create(
            model=MODEL,
            messages=all_messages,
            tools=TOOLS,
        )
        msg = response.choices[0].message

        # Show what the LLM returned
        print(f"\nTurn {turn} - LLM Output:")
        print(f"  content:    {msg.content}")
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  tool_call:  {tc.function.name}({tc.function.arguments})")
        else:
            print(f"  tool_calls: None (done!)")

        # 2. Append assistant turn
        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)

        # 3. If no tool calls, we're done
        if not msg.tool_calls:
            print(f"\n>>> Agent finished in {turn} turn(s)")
            return

        # 4. Execute each tool call, append results
        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            cmd = args.get('command', '')
            print(f"\n  [exec] {name}: {cmd}")
            output = TOOL_HANDLERS[name](**args)
            print(f"  [result] {output[:300]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": output,
            })

### What just happened?

That’s the **entire agent** in under 30 lines of core logic. Let’s trace the flow:

| Step | What happens |
|------|-------------|
| User types a prompt | Added to `messages` as `{"role": "user", ...}` |
| LLM receives messages + tools | Returns either text or tool calls |
| `msg.tool_calls` is not empty | We execute each tool, append results with `role: "tool"` |
| `msg.tool_calls` is empty/None | Model is done — we exit the loop |

The `messages` list grows with each iteration, giving the LLM full context of what it has done so far.

> **Key insight**: Everything else in this course layers on top of this loop — without changing it.

## Try It!

Run the cell below to start an interactive session. Try these prompts:
1. `Create a file called hello.py that prints "Hello, World!"`
2. `List all files in the current directory`
3. `What is the current git branch?`
4. `Create a directory called test_output and write 3 files in it`

Type `q` or `exit` to quit the REPL.

In [ ]:
# Interactive REPL — type your prompts, press Enter. Type "q" to quit.
history = []
while True:
    try:
        query = input("s01 >> ")
    except (EOFError, KeyboardInterrupt):
        break
    if query.strip().lower() in ("q", "exit", ""):
        break
    history.append({"role": "user", "content": query})
    agent_loop(history)
    print()

## create helloWorld.txt and add "Hello world" into it

s01 >>  List all files in the current directory



Turn 1 - LLM Input (2 messages):
  [0] system: You are a coding agent at C:\Users\concao\code\agent101. Use powershell to solve...
  [1] user: List all files in the current directory

Turn 1 - LLM Output:
  content:    {
  "command": "Get-ChildItem -Path ."
}
{
  "command": "Get-ChildItem -Path ."
}
  tool_calls: None (done!)

>>> Agent finished in 1 turn(s)



## What Changed

| Component     | Before     | After                          |
|---------------|------------|--------------------------------|
| Agent loop    | (none)     | `while True` + tool_calls check |
| Tools         | (none)     | `powershell` (one tool)        |
| Messages      | (none)     | Accumulating list              |
| Control flow  | (none)     | Exit when `tool_calls` is empty |

---


# S02:Multiple Tools 

`s01 > [ s02 ] s03 > s04 > s05 > s06 | s07 > s08 > s09 > s10 > s11 > s12`

> *"Adding a tool means adding one handler"* — the loop stays the same; new tools register into the dispatch map.
>
> **Harness layer**: Tool dispatch — expanding what the model can reach.

The Pi agent (OpenClaw) only has **4 default tools**. Claude Code has **~ 20 default tools**. The number of tools defines what the agent can do — but the loop never changes. Adding a tool = adding one handler + one schema entry.

## Problem

With only `powershell`, the agent shells out for everything. Reading files means `Get-Content` which truncates unpredictably. Writing means `Set-Content` which can clobber files without guardrails. Every shell call is an unconstrained security surface.

Dedicated tools like `read_file` and `write_file` let you enforce **path sandboxing** at the tool level.

The key insight: **adding tools does not require changing the loop.**

## Solution

```
+--------+      +-------+      +---------------------------+
|  User  | ---> |  LLM  | ---> | Tool Dispatch             |
| prompt |      |       |      | {                         |
+--------+      +---+---+      |   powershell: run_ps      |
                    ^           |   read_file:  run_read    |
                    |           |   write_file: run_write   |
                    +-----------+   edit_file:  run_edit    |
                    tool_result | }                         |
                                +---------------------------+
```

The dispatch map is a dict: `{tool_name: handler_function}`. One lookup replaces any if/elif chain.

### Step 1: Path Sandboxing

Before adding file tools, we need safety. `safe_path()` prevents the agent from escaping the workspace directory.

In [7]:
from pathlib import Path

WORKDIR = Path.cwd()

def safe_path(p: str) -> Path:
    """Resolve path and ensure it stays within the workspace."""
    path = (WORKDIR / p).resolve()
    if not path.is_relative_to(WORKDIR):
        raise ValueError(f"Path escapes workspace: {p}")
    return path

# Test: safe paths work
print(safe_path("hello.txt"))

# Test: escaping is blocked
try:
    safe_path("../../etc/passwd")
except ValueError as e:
    print(f"Blocked: {e}")

C:\Users\concao\code\agent101\hello.txt
Blocked: Path escapes workspace: ../../etc/passwd


### Step 2: New Tool Handlers

Three new handlers: `read_file`, `write_file`, `edit_file`. Each uses `safe_path()` for sandboxing.

In [8]:
def run_read(path: str, limit: int = None) -> str:
    """Read file contents, optionally limiting line count."""
    try:
        text = safe_path(path).read_text(encoding="utf-8")
        lines = text.splitlines()
        if limit and limit < len(lines):
            lines = lines[:limit] + [f"... ({len(lines) - limit} more lines)"]
        return "\n".join(lines)[:50000]
    except Exception as e:
        return f"Error: {e}"


def run_write(path: str, content: str) -> str:
    """Write content to a file, creating parent directories if needed."""
    try:
        fp = safe_path(path)
        fp.parent.mkdir(parents=True, exist_ok=True)
        fp.write_text(content, encoding="utf-8")
        return f"Wrote {len(content)} bytes to {path}"
    except Exception as e:
        return f"Error: {e}"


def run_edit(path: str, old_text: str, new_text: str) -> str:
    """Replace exact text in a file (first occurrence only)."""
    try:
        fp = safe_path(path)
        content = fp.read_text(encoding="utf-8")
        if old_text not in content:
            return f"Error: Text not found in {path}"
        fp.write_text(content.replace(old_text, new_text, 1), encoding="utf-8")
        return f"Edited {path}"
    except Exception as e:
        return f"Error: {e}"

# Quick test
print(run_write("test_s02.txt", "Hello from S02!"))
print(run_read("test_s02.txt"))
print(run_edit("test_s02.txt", "S02", "Session 02"))
print(run_read("test_s02.txt"))

Wrote 15 bytes to test_s02.txt
Hello from S02!
Edited test_s02.txt
Hello from Session 02!


### Step 3: Tool Schemas & Dispatch Map

Register all 4 tools. The LLM sees the schemas; the dispatch map routes calls to handlers.

In [9]:
TOOLS = [
    {"type": "function", "function": {
        "name": "powershell",
        "description": "Run a PowerShell command.",
        "parameters": {
            "type": "object",
            "properties": {"command": {"type": "string"}},
            "required": ["command"],
        },
    }},
    {"type": "function", "function": {
        "name": "read_file",
        "description": "Read file contents. Use 'limit' to cap line count.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string"},
                "limit": {"type": "integer"},
            },
            "required": ["path"],
        },
    }},
    {"type": "function", "function": {
        "name": "write_file",
        "description": "Write content to a file (creates parent dirs).",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string"},
                "content": {"type": "string"},
            },
            "required": ["path", "content"],
        },
    }},
    {"type": "function", "function": {
        "name": "edit_file",
        "description": "Replace exact text in a file (first occurrence).",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string"},
                "old_text": {"type": "string"},
                "new_text": {"type": "string"},
            },
            "required": ["path", "old_text", "new_text"],
        },
    }},
]

TOOL_HANDLERS = {
    "powershell": run_powershell,
    "read_file":  lambda **kw: run_read(kw["path"], kw.get("limit")),
    "write_file": lambda **kw: run_write(kw["path"], kw["content"]),
    "edit_file":  lambda **kw: run_edit(kw["path"], kw["old_text"], kw["new_text"]),
}

print(f"Registered {len(TOOL_HANDLERS)} tools: {list(TOOL_HANDLERS.keys())}")

Registered 4 tools: ['powershell', 'read_file', 'write_file', 'edit_file']


### Step 4: The Agent Loop (unchanged!)

The loop is **exactly the same** as S01. The only difference is that `TOOL_HANDLERS` now has 4 entries instead of 1.

This is the core design: the loop is generic, tools are pluggable.

In [10]:
def agent_loop(messages: list):
    """Core agent loop: call LLM, execute tools, repeat until done."""
    turn = 0
    while True:
        turn += 1
        all_messages = [{"role": "system", "content": SYSTEM}] + messages

        # Show what we send to the LLM
        print(f"\n{'='*60}")
        print(f"Turn {turn} - LLM Input ({len(all_messages)} messages):")
        print(f"{'='*60}")
        for j, m in enumerate(all_messages):
            role = m["role"]
            if role == "system":
                print(f"  [{j}] system: {m['content'][:80]}...")
            elif role == "user":
                content = m.get("content", "")
                if isinstance(content, str):
                    print(f"  [{j}] user: {content[:100]}")
                else:
                    print(f"  [{j}] user: (tool results)")
            elif role == "assistant":
                tc = m.get("tool_calls")
                if tc:
                    print(f"  [{j}] assistant: (called {len(tc)} tool(s))")
                else:
                    print(f"  [{j}] assistant: {str(m.get('content', ''))[:100]}")
            elif role == "tool":
                print(f"  [{j}] tool: {m['content'][:80]}")

        # 1. Call the LLM
        response = client.chat.completions.create(
            model=MODEL,
            messages=all_messages,
            tools=TOOLS,
        )
        msg = response.choices[0].message

        # Show what the LLM returned
        print(f"\nTurn {turn} - LLM Output:")
        print(f"  content:    {msg.content}")
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  tool_call:  {tc.function.name}({tc.function.arguments})")
        else:
            print(f"  tool_calls: None (done!)")

        # 2. Append assistant turn
        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)

        # 3. If no tool calls, we're done
        if not msg.tool_calls:
            print(f"\n>>> Agent finished in {turn} turn(s)")
            return

        # 4. Execute each tool call, append results
        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            handler = TOOL_HANDLERS.get(name)
            if handler:
                print(f"\n  [exec] {name}: {json.dumps(args)[:100]}")
                output = handler(**args)
            else:
                output = f"Unknown tool: {name}"
            print(f"  [result] {output[:300]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": output,
            })

## Try It!

Now the agent has 4 tools. Try these prompts:
1. `Read the file pyproject.toml`
2. `Create a file called greet.py with a greet(name) function`
3. `Edit greet.py to add a docstring to the function`
4. `Read greet.py to verify the edit worked`

Watch how the LLM picks the right tool for each task — it no longer shells out for everything.

In [11]:
# Interactive REPL — type your prompts, press Enter. Type "q" to quit.
history = []
while True:
    try:
        query = input("s02 >> ")
    except (EOFError, KeyboardInterrupt):
        break
    if query.strip().lower() in ("q", "exit", ""):
        break
    history.append({"role": "user", "content": query})
    agent_loop(history)
    print()

s02 >>  reate a file called greet.py with a greet(name) function



Turn 1 - LLM Input (2 messages):
  [0] system: You are a coding agent at C:\Users\concao\code\agent101. Use powershell to solve...
  [1] user: reate a file called greet.py with a greet(name) function

Turn 1 - LLM Output:
  content:    {
  "command": "New-Item -Path 'C:\\Users\\concao\\code\\agent101\\greet.py' -ItemType File -Force"
}
  tool_call:  write_file({"path":"C:\\Users\\concao\\code\\agent101\\greet.py","content":"def greet(name):\n    print(f\"Hello, {name}!\")\n"})

  [exec] write_file: {"path": "C:\\Users\\concao\\code\\agent101\\greet.py", "content": "def greet(name):\n    print(f\"H
  [result] Wrote 46 bytes to C:\Users\concao\code\agent101\greet.py

Turn 2 - LLM Input (4 messages):
  [0] system: You are a coding agent at C:\Users\concao\code\agent101. Use powershell to solve...
  [1] user: reate a file called greet.py with a greet(name) function
  [2] assistant: (called 1 tool(s))
  [3] tool: Wrote 46 bytes to C:\Users\concao\code\agent101\greet.py

Turn 2 - LLM Output

s02 >>  q


## What Changed From S01

| Component      | Before (S01)           | After (S02)                          |
|----------------|------------------------|--------------------------------------|
| Tools          | 1 (powershell only)    | 4 (powershell, read, write, edit)    |
| Dispatch       | Hardcoded handler      | `TOOL_HANDLERS` dict lookup          |
| Path safety    | None                   | `safe_path()` sandbox                |
| Agent loop     | Unchanged              | **Unchanged**                        |

> **Key insight**: Add a tool = add a handler + add a schema entry. The loop never changes.

---

**Next: Session 03 — TodoWrite** → an agent without a plan drifts. Adding a planning layer with stateful task tracking.

# S03: TodoWrite

`s01 > s02 > [ s03 ] s04 > s05 > s06 | s07 > s08 > s09 > s10 > s11 > s12`

> *"An agent without a plan drifts"* — list the steps first, then execute.
>
> **Harness layer**: Planning — keeping the model on course without scripting the route.

## Problem

On multi-step tasks, the model loses track. It repeats work, skips steps, or wanders off. Long conversations make this worse — the system prompt fades as tool results fill the context. A 10-step refactoring might complete steps 1–3, then the model starts improvising because it forgot steps 4–10.

## Solution

```
+--------+      +-------+      +---------+
|  User  | ---> |  LLM  | ---> | Tools   |
| prompt |      |       |      | + todo  |
+--------+      +---+---+      +----+----+
                    ^                |
                    |   tool_result  |
                    +----------------+
                          |
              +-----------+-----------+
              | TodoManager state     |
              | [ ] task A            |
              | [>] task B  <- doing  |
              | [x] task C            |
              +-----------------------+
                          |
              if rounds_since_todo >= 3:
                inject <reminder>
```

Two mechanisms:
1. **TodoManager** — structured state the LLM writes to. Only one `in_progress` at a time (forces sequential focus).
2. **Nag reminder** — if the model goes 3+ rounds without calling `todo`, inject a nudge into the conversation.

### Step 1: TodoManager

A simple class that stores todo items with statuses: `pending`, `in_progress`, `completed`. Key constraint: **only one task can be `in_progress` at a time** — this forces the model to focus.

In [ ]:
class TodoManager:
    def __init__(self):
        self.items = []

    def update(self, items: list) -> str:
        """Validate and update the todo list."""
        if len(items) > 20:
            raise ValueError("Max 20 todos allowed")
        validated = []
        in_progress_count = 0
        for i, item in enumerate(items):
            text = str(item.get("text", "")).strip()
            status = str(item.get("status", "pending")).lower()
            item_id = str(item.get("id", str(i + 1)))
            if not text:
                raise ValueError(f"Item {item_id}: text required")
            if status not in ("pending", "in_progress", "completed"):
                raise ValueError(f"Item {item_id}: invalid status '{status}'")
            if status == "in_progress":
                in_progress_count += 1
            validated.append({"id": item_id, "text": text, "status": status})
        if in_progress_count > 1:
            raise ValueError("Only one task can be in_progress at a time")
        self.items = validated
        return self.render()

    def render(self) -> str:
        """Display the current todo state."""
        if not self.items:
            return "No todos."
        lines = []
        for item in self.items:
            marker = {"pending": "[ ]", "in_progress": "[>]", "completed": "[x]"}[item["status"]]
            lines.append(f"{marker} #{item['id']}: {item['text']}")
        done = sum(1 for t in self.items if t["status"] == "completed")
        lines.append(f"\n({done}/{len(self.items)} completed)")
        return "\n".join(lines)


TODO = TodoManager()

# Quick test
result = TODO.update([
    {"id": "1", "text": "Read the file", "status": "completed"},
    {"id": "2", "text": "Add type hints", "status": "in_progress"},
    {"id": "3", "text": "Write tests", "status": "pending"},
])
print(result)

### Step 2: Register the `todo` Tool

The `todo` tool goes into the dispatch map like any other tool. Same pattern as S02 — add a handler + add a schema.

In [ ]:
# Reset for a fresh session
TODO = TodoManager()

# Update SYSTEM to mention todo planning
SYSTEM = f"""You are a coding agent at {WORKDIR}.
Use the todo tool to plan multi-step tasks. Mark in_progress before starting, completed when done.
Prefer tools over prose."""

# Add todo to the tool list
TOOLS = [
    {"type": "function", "function": {
        "name": "powershell",
        "description": "Run a PowerShell command.",
        "parameters": {"type": "object", "properties": {"command": {"type": "string"}}, "required": ["command"]},
    }},
    {"type": "function", "function": {
        "name": "read_file",
        "description": "Read file contents. Use 'limit' to cap line count.",
        "parameters": {"type": "object", "properties": {"path": {"type": "string"}, "limit": {"type": "integer"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "write_file",
        "description": "Write content to a file (creates parent dirs).",
        "parameters": {"type": "object", "properties": {"path": {"type": "string"}, "content": {"type": "string"}}, "required": ["path", "content"]},
    }},
    {"type": "function", "function": {
        "name": "edit_file",
        "description": "Replace exact text in a file (first occurrence).",
        "parameters": {"type": "object", "properties": {"path": {"type": "string"}, "old_text": {"type": "string"}, "new_text": {"type": "string"}}, "required": ["path", "old_text", "new_text"]},
    }},
    {"type": "function", "function": {
        "name": "todo",
        "description": "Update task list. Track progress on multi-step tasks. Set status to in_progress before starting, completed when done.",
        "parameters": {
            "type": "object",
            "properties": {
                "items": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "id": {"type": "string"},
                            "text": {"type": "string"},
                            "status": {"type": "string", "enum": ["pending", "in_progress", "completed"]},
                        },
                        "required": ["id", "text", "status"],
                    },
                },
            },
            "required": ["items"],
        },
    }},
]

# Add todo handler to the dispatch map
TOOL_HANDLERS = {
    "powershell": run_powershell,
    "read_file":  lambda **kw: run_read(kw["path"], kw.get("limit")),
    "write_file": lambda **kw: run_write(kw["path"], kw["content"]),
    "edit_file":  lambda **kw: run_edit(kw["path"], kw["old_text"], kw["new_text"]),
    "todo":       lambda **kw: TODO.update(kw["items"]),
}

print(f"Registered {len(TOOL_HANDLERS)} tools: {list(TOOL_HANDLERS.keys())}")

### Step 3: Agent Loop with Nag Reminder

The loop is almost the same as S02, with one addition: a **nag reminder**.

If the model goes 3+ rounds without calling `todo`, we inject `<reminder>Update your todos.</reminder>` into the next message. This creates accountability — the model can’t just forget its plan.

For Azure OpenAI, we inject the reminder as an extra `user` message (since tool results are separate messages, not a list).

In [ ]:
def agent_loop(messages: list):
    """Agent loop with todo nag reminder."""
    turn = 0
    rounds_since_todo = 0
    while True:
        turn += 1
        all_messages = [{"role": "system", "content": SYSTEM}] + messages

        # Inject nag reminder if model forgot to update todos
        if rounds_since_todo >= 3:
            all_messages.append({"role": "user", "content": "<reminder>Update your todos.</reminder>"})
            print(f"  [nag] Injected todo reminder (no todo call for {rounds_since_todo} rounds)")

        # Show turn info
        print(f"\n{'='*60}")
        print(f"Turn {turn} - LLM Input ({len(all_messages)} messages):")
        print(f"{'='*60}")
        for j, m in enumerate(all_messages):
            role = m["role"]
            if role == "system":
                print(f"  [{j}] system: {m['content'][:80]}...")
            elif role == "user":
                content = m.get("content", "")
                if isinstance(content, str):
                    print(f"  [{j}] user: {content[:100]}")
                else:
                    print(f"  [{j}] user: (tool results)")
            elif role == "assistant":
                tc = m.get("tool_calls")
                if tc:
                    print(f"  [{j}] assistant: (called {len(tc)} tool(s))")
                else:
                    print(f"  [{j}] assistant: {str(m.get('content', ''))[:100]}")
            elif role == "tool":
                print(f"  [{j}] tool: {m['content'][:80]}")

        # Call the LLM
        response = client.chat.completions.create(
            model=MODEL,
            messages=all_messages,
            tools=TOOLS,
        )
        msg = response.choices[0].message

        # Show LLM output
        print(f"\nTurn {turn} - LLM Output:")
        print(f"  content:    {msg.content}")
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  tool_call:  {tc.function.name}({tc.function.arguments[:100]})")
        else:
            print(f"  tool_calls: None (done!)")

        # Append assistant turn
        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)

        # If no tool calls, we're done
        if not msg.tool_calls:
            print(f"\n>>> Agent finished in {turn} turn(s)")
            print(f"\n--- Final Todo State ---")
            print(TODO.render())
            return

        # Execute each tool call
        used_todo = False
        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            handler = TOOL_HANDLERS.get(name)
            try:
                output = handler(**args) if handler else f"Unknown tool: {name}"
            except Exception as e:
                output = f"Error: {e}"
            print(f"\n  [exec] {name}: {json.dumps(args)[:100]}")
            print(f"  [result] {str(output)[:300]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(output),
            })
            if name == "todo":
                used_todo = True

        # Track rounds since last todo call
        rounds_since_todo = 0 if used_todo else rounds_since_todo + 1

## Try It!

Give the agent a multi-step task and watch it plan with `todo`:
1. `Create a Python package with __init__.py, utils.py, and tests/test_utils.py`
2. `Refactor greet.py: add type hints, docstrings, and a main guard`
3. `Review all Python files in this directory and fix any style issues`

Watch for:
- The model calling `todo` to create a plan before starting
- Status transitions: `pending` → `in_progress` → `completed`
- The nag reminder if the model forgets to update its todos

In [ ]:
# Interactive REPL — type your prompts, press Enter. Type "q" to quit.
TODO = TodoManager()  # Reset todos for fresh session
history = []
while True:
    try:
        query = input("s03 >> ")
    except (EOFError, KeyboardInterrupt):
        break
    if query.strip().lower() in ("q", "exit", ""):
        break
    history.append({"role": "user", "content": query})
    agent_loop(history)
    print()

## What Changed From S02

| Component      | Before (S02)           | After (S03)                          |
|----------------|------------------------|--------------------------------------|
| Tools          | 4                      | 5 (+todo)                            |
| Planning       | None                   | TodoManager with statuses            |
| Nag injection  | None                   | `<reminder>` after 3 rounds          |
| Agent loop     | Simple dispatch        | + rounds_since_todo counter          |

> **Key insight**: The model can track its own progress — and you can see it. The "one in_progress" constraint forces sequential focus. The nag creates accountability.

---

**Next: Session 04 — Subagent** → break big tasks down; each subtask gets a clean context.

# S04: Subagents

`s01 > s02 > s03 > [ s04 ] s05 > s06 | s07 > s08 > s09 > s10 > s11 > s12`

> *"Break big tasks down; each subtask gets a clean context"*
>
> **Harness layer**: Context isolation — child agents run in fresh message lists.

## Problem

Complex tasks pollute the main context. A refactoring that requires reading 10 files, planning, and writing code fills the message list with tool results. By turn 30, the model is confused by its own history. Subtasks deserve a **clean slate** — and the parent only needs the final answer.

## Solution

```
+--------+      +-------+      +---------+
|  User  | ---> |  LLM  | ---> |  task   |---> run_subagent(prompt)
| prompt |      |       |      |  tool   |         |
+--------+      +---+---+      +----+----+    +----v-----+
                    ^                |         | Child LLM|
                    |   final text   |         | fresh [] |
                    +----------------+         | no task  |
                                               +----------+
```

The parent gets a `task` tool. The child gets all base tools **except** `task` (prevents recursion). Only the child's final text bubbles up.

### Step 1: Subagent System Prompt & Child Tools

The child agent gets a focused system prompt and the base tools (no `task` to prevent infinite spawning).

In [ ]:
SUBAGENT_SYSTEM = (
    f"You are a sub-agent working in {WORKDIR}. "
    "Solve the given task using your tools. Be concise and thorough. "
    "When done, reply with a text summary of what you did."
)

# Child gets all base tools except 'task' (no recursion)
CHILD_TOOLS = [
    {"type": "function", "function": {
        "name": "powershell",
        "description": "Run a PowerShell command.",
        "parameters": {"type": "object", "properties": {"command": {"type": "string"}}, "required": ["command"]},
    }},
    {"type": "function", "function": {
        "name": "read_file",
        "description": "Read file contents. Use 'limit' to cap line count.",
        "parameters": {"type": "object", "properties": {"path": {"type": "string"}, "limit": {"type": "integer"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "write_file",
        "description": "Write content to a file (creates parent dirs).",
        "parameters": {"type": "object", "properties": {"path": {"type": "string"}, "content": {"type": "string"}}, "required": ["path", "content"]},
    }},
    {"type": "function", "function": {
        "name": "edit_file",
        "description": "Replace exact text in a file (first occurrence).",
        "parameters": {"type": "object", "properties": {"path": {"type": "string"}, "old_text": {"type": "string"}, "new_text": {"type": "string"}}, "required": ["path", "old_text", "new_text"]},
    }},
    {"type": "function", "function": {
        "name": "todo",
        "description": "Update task list. Track progress on multi-step tasks.",
        "parameters": {
            "type": "object",
            "properties": {
                "items": {"type": "array", "items": {"type": "object", "properties": {
                    "id": {"type": "string"}, "text": {"type": "string"},
                    "status": {"type": "string", "enum": ["pending", "in_progress", "completed"]}
                }, "required": ["id", "text", "status"]}},
            },
            "required": ["items"],
        },
    }},
]

# Child tool handlers (same as parent minus 'task')
CHILD_HANDLERS = {
    "powershell": run_powershell,
    "read_file":  lambda **kw: run_read(kw["path"], kw.get("limit")),
    "write_file": lambda **kw: run_write(kw["path"], kw["content"]),
    "edit_file":  lambda **kw: run_edit(kw["path"], kw["old_text"], kw["new_text"]),
    "todo":       lambda **kw: TODO.update(kw["items"]),
}

print(f"Child tools: {[t['function']['name'] for t in CHILD_TOOLS]}")

### Step 2: The Subagent Runner

`run_subagent(prompt)` spawns a child with **fresh messages**. It runs its own tool loop (max 30 iterations) and returns only the final text to the parent.

In [ ]:
def run_subagent(prompt: str) -> str:
    '''Spawn a child agent with fresh context. Returns only final text.'''
    print(f"\n  [subagent] Starting child for: {prompt[:80]}...")
    sub_messages = [{"role": "user", "content": prompt}]
    for i in range(30):
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "system", "content": SUBAGENT_SYSTEM}] + sub_messages,
            tools=CHILD_TOOLS,
        )
        msg = response.choices[0].message
        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        sub_messages.append(assistant_msg)
        if not msg.tool_calls:
            print(f"  [subagent] Done in {i+1} turn(s)")
            break
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            handler = CHILD_HANDLERS.get(name)
            try:
                output = handler(**args) if handler else f"Unknown tool: {name}"
            except Exception as e:
                output = f"Error: {e}"
            print(f"    [child-exec] {name}: {json.dumps(args)[:80]}")
            sub_messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(output)[:50000]})
    return msg.content or "(no summary)"

print("run_subagent() defined")

### Step 3: Parent Tools & Dispatch Map

The parent gets all child tools **plus** the `task` tool. When the LLM calls `task`, it dispatches to `run_subagent()`.

In [ ]:
# Parent tools = child tools + task
TASK_TOOL = {"type": "function", "function": {
    "name": "task",
    "description": "Delegate a subtask to a child agent with clean context. Returns the child's summary.",
    "parameters": {
        "type": "object",
        "properties": {
            "prompt": {"type": "string", "description": "The task for the child agent to solve."},
        },
        "required": ["prompt"],
    },
}}

TOOLS = CHILD_TOOLS + [TASK_TOOL]

TOOL_HANDLERS = {
    **CHILD_HANDLERS,
    "task": lambda **kw: run_subagent(kw["prompt"]),
}

SYSTEM = (
    f"You are a coding agent at {WORKDIR}. "
    "Use the 'task' tool to delegate complex subtasks to child agents. "
    "Each child gets a clean context. Use todo to plan, then delegate. "
    "Prefer tools over prose."
)

TODO = TodoManager()
print(f"Parent tools: {[t['function']['name'] for t in TOOLS]}")

### Step 4: Agent Loop (with subagent support)

The loop is the same structure — the `task` tool handler does all the subagent work. The loop just dispatches as usual.

In [ ]:
def agent_loop(messages: list):
    '''Agent loop with subagent support via task tool.'''
    turn = 0
    rounds_since_todo = 0
    while True:
        turn += 1
        all_messages = [{"role": "system", "content": SYSTEM}] + messages

        # Nag reminder from S03
        if rounds_since_todo >= 3:
            all_messages.append({"role": "user", "content": "<reminder>Update your todos.</reminder>"})
            print(f"  [nag] Injected todo reminder")

        # Show turn info
        print(f"\n{'='*60}")
        print(f"Turn {turn} - LLM Input ({len(all_messages)} messages):")
        print(f"{'='*60}")
        for j, m in enumerate(all_messages):
            role = m["role"]
            if role == "system":
                print(f"  [{j}] system: {m['content'][:80]}...")
            elif role == "user":
                content = m.get("content", "")
                if isinstance(content, str):
                    print(f"  [{j}] user: {content[:100]}")
                else:
                    print(f"  [{j}] user: (tool results)")
            elif role == "assistant":
                tc = m.get("tool_calls")
                if tc:
                    print(f"  [{j}] assistant: (called {len(tc)} tool(s))")
                else:
                    print(f"  [{j}] assistant: {str(m.get('content', ''))[:100]}")
            elif role == "tool":
                print(f"  [{j}] tool: {m['content'][:80]}")

        # Call the LLM
        response = client.chat.completions.create(
            model=MODEL,
            messages=all_messages,
            tools=TOOLS,
        )
        msg = response.choices[0].message

        # Show LLM output
        print(f"\nTurn {turn} - LLM Output:")
        print(f"  content:    {msg.content}")
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  tool_call:  {tc.function.name}({tc.function.arguments[:100]})")
        else:
            print(f"  tool_calls: None (done!)")

        # Append assistant turn
        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)

        # If no tool calls, we're done
        if not msg.tool_calls:
            print(f"\n>>> Agent finished in {turn} turn(s)")
            return

        # Execute each tool call
        used_todo = False
        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            handler = TOOL_HANDLERS.get(name)
            try:
                output = handler(**args) if handler else f"Unknown tool: {name}"
            except Exception as e:
                output = f"Error: {e}"
            print(f"\n  [exec] {name}: {json.dumps(args)[:100]}")
            print(f"  [result] {str(output)[:300]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(output),
            })
            if name == "todo":
                used_todo = True

        rounds_since_todo = 0 if used_todo else rounds_since_todo + 1

## Try It!

Give the agent a task that benefits from delegation:
1. `Create 3 Python files: utils.py, math_helpers.py, and string_helpers.py with useful functions in each`
2. `Read all .py files in this directory and create a summary document`

Watch the agent use `task` to spawn child agents with clean contexts.

In [ ]:
# Interactive REPL
TODO = TodoManager()
history = []
while True:
    try:
        query = input("s04 >> ")
    except (EOFError, KeyboardInterrupt):
        break
    if query.strip().lower() in ("q", "exit", ""):
        break
    history.append({"role": "user", "content": query})
    agent_loop(history)
    print()

## What Changed From S03

| Component      | Before (S03)           | After (S04)                          |
|----------------|------------------------|--------------------------------------|
| Tools          | 5                      | 6 (+task)                            |
| Subagent       | None                   | `run_subagent()` with fresh context  |
| Child isolation | N/A                   | CHILD_TOOLS (no task = no recursion) |
| Agent loop     | + todo nag             | + subagent dispatch (via handler)    |

> **Key insight**: The child agent gets a clean `messages=[]`. Only its final text returns to the parent. This keeps the parent's context clean.

---

**Next: Session 05 — Skill Loading** → load knowledge when you need it, not upfront.

# S05: Skill Loading

`s01 > s02 > s03 > s04 > [ s05 ] s06 | s07 > s08 > s09 > s10 > s11 > s12`

> *"Load knowledge when you need it, not upfront"*
>
> **Harness layer**: On-demand knowledge — two-layer injection keeps the system prompt small.

## Problem

Stuffing all domain knowledge into the system prompt wastes tokens and dilutes focus. A 5000-token system prompt about code review, testing patterns, and API conventions means the model always pays the cost even when only doing a simple file rename. We need knowledge **on demand**.

## Solution

```
+------------------+
| System Prompt    |       skills/
|  ...             |       +-- code-review/
|  Available:      |       |   +-- SKILL.md   (~2000 tokens)
|  - code-review   | <--+  +-- testing/
|    (~100 tokens)  |    |     +-- SKILL.md
+------------------+    |
                        |
  load_skill("code-review")
     |
     +-> injects full body into context
```

**Layer 1**: Skill names + short descriptions in system prompt (~100 tokens each).
**Layer 2**: Full skill body loaded via `load_skill` tool (~2000 tokens, on demand).

### Step 1: Create a Sample Skill

Skills live in `skills/<name>/SKILL.md` with YAML frontmatter. Let's create one for testing.

In [ ]:
import yaml

# Create skills directory and a sample skill
skill_dir = WORKDIR / "skills" / "code-review"
skill_dir.mkdir(parents=True, exist_ok=True)

skill_content = (
    "---\n"
    "name: code-review\n"
    'description: "Guidelines for reviewing Python code: style, bugs, security."\n'
    "---\n"
    "\n"
    "# Code Review Skill\n"
    "\n"
    "When reviewing Python code, check for:\n"
    "\n"
    "## Style\n"
    "- PEP 8 compliance (naming, spacing, line length)\n"
    "- Consistent use of type hints\n"
    "- Docstrings on public functions\n"
    "\n"
    "## Bugs\n"
    "- Off-by-one errors in loops\n"
    "- Unclosed file handles (use context managers)\n"
    "- Mutable default arguments\n"
    "\n"
    "## Security\n"
    "- No hardcoded secrets or API keys\n"
    "- Input validation on user-supplied data\n"
    "- Safe use of subprocess (no shell=True with user input)\n"
)

(skill_dir / "SKILL.md").write_text(skill_content, encoding="utf-8")
print(f"Created {skill_dir / 'SKILL.md'}")

### Step 2: SkillLoader Class

Scans `skills/*/SKILL.md`, parses YAML frontmatter. `get_descriptions()` returns a short list for the system prompt. `get_content(name)` returns the full skill body wrapped in `<skill>` tags.

In [ ]:
class SkillLoader:
    def __init__(self, skills_dir):
        self.skills_dir = Path(skills_dir)
        self.skills = {}
        self._scan()

    def _scan(self):
        '''Scan skills directory for SKILL.md files with YAML frontmatter.'''
        if not self.skills_dir.exists():
            return
        for skill_file in self.skills_dir.glob("*/SKILL.md"):
            text = skill_file.read_text(encoding="utf-8")
            # Parse YAML frontmatter between --- markers
            if text.startswith("---"):
                parts = text.split("---", 2)
                if len(parts) >= 3:
                    meta = yaml.safe_load(parts[1])
                    body = parts[2].strip()
                    name = meta.get("name", skill_file.parent.name)
                    self.skills[name] = {
                        "description": meta.get("description", ""),
                        "body": body,
                    }

    def get_descriptions(self) -> str:
        '''Short descriptions for the system prompt (~100 tokens each).'''
        if not self.skills:
            return "No skills loaded."
        lines = ["Available skills (use load_skill to get full details):"]
        for name, info in self.skills.items():
            lines.append(f"  - {name}: {info['description']}")
        return "\n".join(lines)

    def get_content(self, name: str) -> str:
        '''Full skill body wrapped in <skill> tags.'''
        skill = self.skills.get(name)
        if not skill:
            available = ", ".join(self.skills.keys()) or "none"
            return f"Skill '{name}' not found. Available: {available}"
        return f"<skill name='{name}'>\n{skill['body']}\n</skill>"


SKILLS = SkillLoader(WORKDIR / "skills")
print("Loaded skills:")
print(SKILLS.get_descriptions())

### Step 3: Register `load_skill` Tool & Update System Prompt

Skill descriptions go into the system prompt (Layer 1). The `load_skill` tool provides full content on demand (Layer 2).

In [ ]:
SYSTEM = (
    f"You are a coding agent at {WORKDIR}.\n"
    f"Use todo to plan multi-step tasks. Prefer tools over prose.\n\n"
    f"{SKILLS.get_descriptions()}"
)

TOOLS = CHILD_TOOLS + [
    TASK_TOOL,
    {"type": "function", "function": {
        "name": "load_skill",
        "description": "Load full content of a skill by name. Use when you need domain-specific guidance.",
        "parameters": {
            "type": "object",
            "properties": {"name": {"type": "string", "description": "Skill name to load."}},
            "required": ["name"],
        },
    }},
]

TOOL_HANDLERS = {
    **CHILD_HANDLERS,
    "task":       lambda **kw: run_subagent(kw["prompt"]),
    "load_skill": lambda **kw: SKILLS.get_content(kw["name"]),
}

TODO = TodoManager()
print(f"Tools: {[t['function']['name'] for t in TOOLS]}")
print(f"\nSystem prompt ({len(SYSTEM)} chars):")
print(SYSTEM[:300])

### Step 4: Agent Loop

Same loop as S04 — the `load_skill` handler does the work. No loop changes needed for skill loading.

In [ ]:
def agent_loop(messages: list):
    '''Agent loop with skill loading + subagent support.'''
    turn = 0
    rounds_since_todo = 0
    while True:
        turn += 1
        all_messages = [{"role": "system", "content": SYSTEM}] + messages

        if rounds_since_todo >= 3:
            all_messages.append({"role": "user", "content": "<reminder>Update your todos.</reminder>"})
            print(f"  [nag] Injected todo reminder")

        print(f"\n{'='*60}")
        print(f"Turn {turn} - LLM Input ({len(all_messages)} messages):")
        print(f"{'='*60}")
        for j, m in enumerate(all_messages):
            role = m["role"]
            if role == "system":
                print(f"  [{j}] system: {m['content'][:80]}...")
            elif role == "user":
                content = m.get("content", "")
                if isinstance(content, str):
                    print(f"  [{j}] user: {content[:100]}")
                else:
                    print(f"  [{j}] user: (tool results)")
            elif role == "assistant":
                tc = m.get("tool_calls")
                if tc:
                    print(f"  [{j}] assistant: (called {len(tc)} tool(s))")
                else:
                    print(f"  [{j}] assistant: {str(m.get('content', ''))[:100]}")
            elif role == "tool":
                print(f"  [{j}] tool: {m['content'][:80]}")

        response = client.chat.completions.create(
            model=MODEL,
            messages=all_messages,
            tools=TOOLS,
        )
        msg = response.choices[0].message

        print(f"\nTurn {turn} - LLM Output:")
        print(f"  content:    {msg.content}")
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  tool_call:  {tc.function.name}({tc.function.arguments[:100]})")
        else:
            print(f"  tool_calls: None (done!)")

        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)

        if not msg.tool_calls:
            print(f"\n>>> Agent finished in {turn} turn(s)")
            return

        used_todo = False
        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            handler = TOOL_HANDLERS.get(name)
            try:
                output = handler(**args) if handler else f"Unknown tool: {name}"
            except Exception as e:
                output = f"Error: {e}"
            print(f"\n  [exec] {name}: {json.dumps(args)[:100]}")
            print(f"  [result] {str(output)[:300]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(output),
            })
            if name == "todo":
                used_todo = True

        rounds_since_todo = 0 if used_todo else rounds_since_todo + 1

## Try It!

Try a task that triggers skill loading:
1. `Review the greet.py file for code quality issues`
2. `Load the code-review skill and then check pyproject.toml`

Watch the agent load the skill on demand instead of having it in the system prompt always.

In [ ]:
# Interactive REPL
TODO = TodoManager()
SKILLS = SkillLoader(WORKDIR / "skills")  # Rescan
history = []
while True:
    try:
        query = input("s05 >> ")
    except (EOFError, KeyboardInterrupt):
        break
    if query.strip().lower() in ("q", "exit", ""):
        break
    history.append({"role": "user", "content": query})
    agent_loop(history)
    print()

## What Changed From S04

| Component      | Before (S04)           | After (S05)                              |
|----------------|------------------------|------------------------------------------|
| Tools          | 6                      | 7 (+load_skill)                          |
| Knowledge      | All in system prompt   | Layer 1: names, Layer 2: on-demand body  |
| SkillLoader    | None                   | Scans skills/*/SKILL.md, serves content  |
| System prompt  | Static                 | Dynamic with skill descriptions          |

> **Key insight**: ~100 tokens per skill in the system prompt vs ~2000 tokens loaded on demand. With 20 skills, that saves ~38,000 tokens per turn.

---

**Next: Session 06 — Context Compact** → context will fill up; you need a way to make room.

# S06: Context Compact

`s01 > s02 > s03 > s04 > s05 > [ s06 ] | s07 > s08 > s09 > s10 > s11 > s12`

> *"Context will fill up; you need a way to make room"*
>
> **Harness layer**: Compression for infinite sessions — three layers of context management.

## Problem

Every tool call adds to the message list. After 20+ turns, the context is full of stale tool results — file contents read 15 turns ago, shell output from completed tasks. The model pays attention to all of it, wasting tokens and diluting focus. Eventually you hit the context limit and the session crashes.

## Solution

```
Three compression layers:

Layer 1 -- micro_compact (every turn)
  Replace tool results >3 turns old with placeholders
  Cost: 0 LLM calls

Layer 2 -- auto_compact (when tokens > threshold)
  Save transcript to disk, LLM summarizes, replace messages
  Cost: 1 LLM call

Layer 3 -- compact tool (model decides)
  Model explicitly calls compact when it feels context is heavy
  Cost: 1 LLM call
```

Each layer is progressively more aggressive. Layer 1 runs silently every turn. Layer 2 kicks in automatically. Layer 3 gives the model agency.

### Step 1: Token Estimation

A quick-and-dirty estimate: `len(json.dumps(messages)) / 4`. Not accurate, but good enough for threshold checks.

In [ ]:
import time

# Directory for saving transcripts before compaction
TRANSCRIPT_DIR = WORKDIR / ".transcripts"
TRANSCRIPT_DIR.mkdir(exist_ok=True)

TOKEN_THRESHOLD = 40000  # Auto-compact when estimated tokens exceed this

def estimate_tokens(messages: list) -> int:
    '''Rough token estimate: chars / 4.'''
    return len(json.dumps(messages, default=str)) // 4

# Test
sample = [{"role": "user", "content": "hello " * 1000}]
print(f"Sample message tokens (est): {estimate_tokens(sample)}")

### Step 2: Micro-Compact (Layer 1)

Every turn, replace tool result content older than 3 turns with a short placeholder. This is free (no LLM calls) and keeps the context from growing linearly.

In [ ]:
def micro_compact(messages: list) -> list:
    '''Replace old tool results (>3 turns old) with placeholders.'''
    if len(messages) <= 6:
        return messages
    cutoff = len(messages) - 6  # Keep last 3 pairs (assistant + tool)
    result = []
    for i, msg in enumerate(messages):
        if i < cutoff and msg.get("role") == "tool":
            # Find which tool was called by looking at preceding assistant msg
            tool_name = "tool"
            for j in range(i - 1, -1, -1):
                tc_list = messages[j].get("tool_calls", [])
                for tc in tc_list:
                    if tc.get("id") == msg.get("tool_call_id"):
                        tool_name = tc["function"]["name"]
                        break
            result.append({
                "role": "tool",
                "tool_call_id": msg.get("tool_call_id", ""),
                "content": f"[Previous: used {tool_name}]",
            })
        else:
            result.append(msg)
    return result

print("micro_compact() defined")

### Step 3: Auto-Compact (Layer 2) & Compact Tool (Layer 3)

When tokens exceed the threshold, save the full transcript to disk, ask the LLM to summarize, and replace the entire message list with the summary. The `compact` tool lets the model trigger this explicitly.

In [ ]:
def auto_compact(messages: list) -> list:
    '''Save transcript, LLM summarizes, return compressed messages.'''
    # Save transcript to disk
    transcript_path = TRANSCRIPT_DIR / f"transcript_{int(time.time())}.jsonl"
    with open(transcript_path, "w", encoding="utf-8") as f:
        for msg in messages:
            f.write(json.dumps(msg, default=str) + "\n")
    print(f"  [compact] Saved transcript to {transcript_path.name}")

    # LLM summarizes
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "Summarize this conversation for continuity. "
                "Focus on what was accomplished and what's pending."
            )},
            {"role": "user", "content": json.dumps(messages, default=str)[:80000]},
        ],
    )
    summary = response.choices[0].message.content
    print(f"  [compact] Compressed {len(messages)} messages into summary")
    return [{"role": "user", "content": f"[Compressed context]\n\n{summary}"}]


def run_compact(**kwargs) -> str:
    '''Tool handler: compact is handled in the agent loop (returns placeholder).'''
    return "Compaction requested. Will apply before next turn."

print("auto_compact() and run_compact() defined")

### Step 4: Updated Tools & Agent Loop

The agent loop now runs all 3 compression layers:
1. `micro_compact` every turn (Layer 1)
2. `auto_compact` when tokens > threshold (Layer 2)
3. `compact` tool call triggers Layer 2 explicitly (Layer 3)

In [ ]:
TOOLS = CHILD_TOOLS + [
    TASK_TOOL,
    {"type": "function", "function": {
        "name": "load_skill",
        "description": "Load full content of a skill by name.",
        "parameters": {"type": "object", "properties": {"name": {"type": "string"}}, "required": ["name"]},
    }},
    {"type": "function", "function": {
        "name": "compact",
        "description": "Compress conversation context. Use when context feels heavy or you're doing many tool calls.",
        "parameters": {"type": "object", "properties": {}},
    }},
]

TOOL_HANDLERS = {
    **CHILD_HANDLERS,
    "task":       lambda **kw: run_subagent(kw["prompt"]),
    "load_skill": lambda **kw: SKILLS.get_content(kw["name"]),
    "compact":    run_compact,
}

SYSTEM = (
    f"You are a coding agent at {WORKDIR}.\n"
    "Use todo to plan multi-step tasks. Prefer tools over prose.\n"
    "Use compact when context feels heavy after many tool calls.\n\n"
    f"{SKILLS.get_descriptions()}"
)

TODO = TodoManager()
print(f"Tools: {[t['function']['name'] for t in TOOLS]}")

In [ ]:
def agent_loop(messages: list):
    '''Agent loop with 3-layer context compaction.'''
    turn = 0
    rounds_since_todo = 0
    compact_requested = False
    while True:
        turn += 1

        # Layer 1: micro-compact old tool results
        messages[:] = micro_compact(messages)

        # Layer 2: auto-compact if over token threshold
        est = estimate_tokens(messages)
        if est > TOKEN_THRESHOLD or compact_requested:
            reason = "tool request" if compact_requested else f"tokens ({est}) > {TOKEN_THRESHOLD}"
            print(f"  [compact] Triggering auto_compact: {reason}")
            messages[:] = auto_compact(messages)
            compact_requested = False

        all_messages = [{"role": "system", "content": SYSTEM}] + messages

        if rounds_since_todo >= 3:
            all_messages.append({"role": "user", "content": "<reminder>Update your todos.</reminder>"})
            print(f"  [nag] Injected todo reminder")

        # Show turn info
        print(f"\n{'='*60}")
        print(f"Turn {turn} - LLM Input ({len(all_messages)} msgs, ~{estimate_tokens(all_messages)} tokens):")
        print(f"{'='*60}")
        for j, m in enumerate(all_messages):
            role = m["role"]
            if role == "system":
                print(f"  [{j}] system: {m['content'][:80]}...")
            elif role == "user":
                content = m.get("content", "")
                if isinstance(content, str):
                    print(f"  [{j}] user: {content[:100]}")
                else:
                    print(f"  [{j}] user: (tool results)")
            elif role == "assistant":
                tc = m.get("tool_calls")
                if tc:
                    print(f"  [{j}] assistant: (called {len(tc)} tool(s))")
                else:
                    print(f"  [{j}] assistant: {str(m.get('content', ''))[:100]}")
            elif role == "tool":
                print(f"  [{j}] tool: {m['content'][:80]}")

        response = client.chat.completions.create(
            model=MODEL,
            messages=all_messages,
            tools=TOOLS,
        )
        msg = response.choices[0].message

        print(f"\nTurn {turn} - LLM Output:")
        print(f"  content:    {msg.content}")
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  tool_call:  {tc.function.name}({tc.function.arguments[:100]})")
        else:
            print(f"  tool_calls: None (done!)")

        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)

        if not msg.tool_calls:
            print(f"\n>>> Agent finished in {turn} turn(s)")
            return

        used_todo = False
        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            handler = TOOL_HANDLERS.get(name)
            try:
                output = handler(**args) if handler else f"Unknown tool: {name}"
            except Exception as e:
                output = f"Error: {e}"
            print(f"\n  [exec] {name}: {json.dumps(args)[:100]}")
            print(f"  [result] {str(output)[:300]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(output),
            })
            if name == "todo":
                used_todo = True
            if name == "compact":
                compact_requested = True

        rounds_since_todo = 0 if used_todo else rounds_since_todo + 1

## Try It!

Try a long session and watch compaction kick in:
1. `Read every .py file in this directory and summarize each one` (generates lots of tool results)
2. `compact` (model triggers explicit compaction)
3. `What files did you just read?` (tests that the summary preserved context)

Watch the token estimates in the turn headers.

In [ ]:
# Interactive REPL
TODO = TodoManager()
history = []
while True:
    try:
        query = input("s06 >> ")
    except (EOFError, KeyboardInterrupt):
        break
    if query.strip().lower() in ("q", "exit", ""):
        break
    history.append({"role": "user", "content": query})
    agent_loop(history)
    print()

## What Changed From S05

| Component      | Before (S05)           | After (S06)                              |
|----------------|------------------------|------------------------------------------|
| Tools          | 7                      | 8 (+compact)                             |
| Token tracking | None                   | `estimate_tokens()` per turn             |
| Layer 1        | None                   | `micro_compact` — free, every turn        |
| Layer 2        | None                   | `auto_compact` — 1 LLM call at threshold  |
| Layer 3        | None                   | `compact` tool — model-initiated          |
| Transcripts    | None                   | Saved to `.transcripts/` before compact  |

> **Key insight**: Compression is not optional for long sessions. Layer 1 is free and always on. Layer 2 is automatic insurance. Layer 3 gives the model agency over its own context.

---

**Next: Session 07 — Task System** → break big goals into small tasks, order them, persist to disk.

# S07: Task System

`s01 > s02 > s03 > s04 > s05 > s06 | [ s07 ] s08 > s09 > s10 > s11 > s12`

> *"Break big goals into small tasks, order them, persist to disk"*
>
> **Harness layer**: Persistent tasks — a file-based task graph that survives restarts.

## Problem

The S03 `TodoManager` is in-memory — restart the kernel and it's gone. It also has no concept of dependencies: task B should wait for task A, but the model has no way to express that. For complex projects with blocking dependencies, we need **persistent, structured tasks**.

## Solution

```
.tasks/
 +-- task_001.json   {"id": 1, "title": "...", "status": "done", "blockedBy": []}
 +-- task_002.json   {"id": 2, "title": "...", "status": "ready", "blockedBy": []}
 +-- task_003.json   {"id": 3, "title": "...", "status": "blocked", "blockedBy": [2]}

When task 2 completes -> auto-clear '2' from task 3's blockedBy -> task 3 becomes ready
```

Each task is a JSON file. Completing a task auto-clears it from dependents' `blockedBy` lists. Tasks survive kernel restarts.

### Step 1: TaskManager Class

File-based task management with dependency tracking. Each task gets its own JSON file in `.tasks/`.

In [ ]:
class TaskManager:
    '''File-based task graph with dependencies.'''

    def __init__(self, tasks_dir=None):
        self.tasks_dir = Path(tasks_dir) if tasks_dir else WORKDIR / ".tasks"
        self.tasks_dir.mkdir(exist_ok=True)
        self._next_id = self._get_next_id()

    def _get_next_id(self) -> int:
        ids = []
        for f in self.tasks_dir.glob("task_*.json"):
            try:
                ids.append(int(f.stem.split("_")[1]))
            except (ValueError, IndexError):
                pass
        return max(ids, default=0) + 1

    def _task_path(self, task_id: int) -> Path:
        return self.tasks_dir / f"task_{task_id:03d}.json"

    def _read_task(self, task_id: int) -> dict:
        path = self._task_path(task_id)
        if not path.exists():
            return None
        return json.loads(path.read_text(encoding="utf-8"))

    def _write_task(self, task: dict):
        path = self._task_path(task["id"])
        path.write_text(json.dumps(task, indent=2), encoding="utf-8")

    def create(self, title: str, description: str = "", blocked_by: list = None) -> str:
        '''Create a new task. Returns confirmation.'''
        task = {
            "id": self._next_id,
            "title": title,
            "description": description,
            "status": "blocked" if blocked_by else "ready",
            "blockedBy": blocked_by or [],
        }
        self._write_task(task)
        self._next_id += 1
        return f"Created task #{task['id']}: {title} (status: {task['status']})"

    def get(self, task_id: int) -> str:
        '''Get task details.'''
        task = self._read_task(task_id)
        if not task:
            return f"Task #{task_id} not found."
        return json.dumps(task, indent=2)

    def update(self, task_id: int, status: str) -> str:
        '''Update task status. Completing a task clears dependencies.'''
        task = self._read_task(task_id)
        if not task:
            return f"Task #{task_id} not found."
        old_status = task["status"]
        task["status"] = status
        self._write_task(task)
        result = f"Task #{task_id}: {old_status} -> {status}"
        # Auto-clear from dependents when completed
        if status == "done":
            cleared = self._clear_dependency(task_id)
            if cleared:
                result += f" (unblocked: {cleared})"
        return result

    def _clear_dependency(self, completed_id: int) -> list:
        '''Remove completed_id from all dependents blockedBy lists.'''
        unblocked = []
        for f in self.tasks_dir.glob("task_*.json"):
            t = json.loads(f.read_text(encoding="utf-8"))
            if completed_id in t.get("blockedBy", []):
                t["blockedBy"].remove(completed_id)
                if not t["blockedBy"] and t["status"] == "blocked":
                    t["status"] = "ready"
                    unblocked.append(t["id"])
                self._write_task(t)
        return unblocked

    def list_all(self) -> str:
        '''List all tasks with their status.'''
        tasks = []
        for f in sorted(self.tasks_dir.glob("task_*.json")):
            tasks.append(json.loads(f.read_text(encoding="utf-8")))
        if not tasks:
            return "No tasks."
        lines = []
        icons = {"ready": "[ ]", "in_progress": "[>]", "done": "[x]", "blocked": "[!]"}
        for t in tasks:
            icon = icons.get(t['status'], '[?]')
            blocked = f" (blocked by: {t['blockedBy']})" if t.get('blockedBy') else ""
            lines.append(f"{icon} #{t['id']}: {t['title']}{blocked}")
        done = sum(1 for t in tasks if t['status'] == 'done')
        lines.append(f"\n({done}/{len(tasks)} done)")
        return "\n".join(lines)


TASKS = TaskManager()

# Quick test
print(TASKS.create("Set up project structure"))
print(TASKS.create("Write core module", blocked_by=[1]))
print(TASKS.create("Write tests", blocked_by=[2]))
print()
print(TASKS.list_all())
print()
print(TASKS.update(1, "done"))
print()
print(TASKS.list_all())

### Step 2: Task Tools & Updated Dispatch

Four new tools: `task_create`, `task_update`, `task_list`, `task_get`. These give the model full control over the persistent task graph.

In [ ]:
# Reset TaskManager for the session
TASKS = TaskManager()

TASK_MGMT_TOOLS = [
    {"type": "function", "function": {
        "name": "task_create",
        "description": "Create a persistent task. Use blocked_by for dependencies.",
        "parameters": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "description": {"type": "string"},
                "blocked_by": {"type": "array", "items": {"type": "integer"}, "description": "Task IDs this depends on."},
            },
            "required": ["title"],
        },
    }},
    {"type": "function", "function": {
        "name": "task_update",
        "description": "Update task status (ready, in_progress, done). Completing auto-unblocks dependents.",
        "parameters": {
            "type": "object",
            "properties": {
                "task_id": {"type": "integer"},
                "status": {"type": "string", "enum": ["ready", "in_progress", "done"]},
            },
            "required": ["task_id", "status"],
        },
    }},
    {"type": "function", "function": {
        "name": "task_list",
        "description": "List all tasks with status and dependencies.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "task_get",
        "description": "Get full details of a specific task.",
        "parameters": {
            "type": "object",
            "properties": {"task_id": {"type": "integer"}},
            "required": ["task_id"],
        },
    }},
]

TOOLS = CHILD_TOOLS + [
    TASK_TOOL,
    {"type": "function", "function": {
        "name": "load_skill",
        "description": "Load full content of a skill by name.",
        "parameters": {"type": "object", "properties": {"name": {"type": "string"}}, "required": ["name"]},
    }},
    {"type": "function", "function": {
        "name": "compact",
        "description": "Compress conversation context.",
        "parameters": {"type": "object", "properties": {}},
    }},
] + TASK_MGMT_TOOLS

TOOL_HANDLERS = {
    **CHILD_HANDLERS,
    "task":        lambda **kw: run_subagent(kw["prompt"]),
    "load_skill":  lambda **kw: SKILLS.get_content(kw["name"]),
    "compact":     run_compact,
    "task_create": lambda **kw: TASKS.create(kw["title"], kw.get("description", ""), kw.get("blocked_by")),
    "task_update": lambda **kw: TASKS.update(kw["task_id"], kw["status"]),
    "task_list":   lambda **kw: TASKS.list_all(),
    "task_get":    lambda **kw: TASKS.get(kw["task_id"]),
}

SYSTEM = (
    f"You are a coding agent at {WORKDIR}.\n"
    "Use task_create/task_update/task_list to manage persistent tasks with dependencies.\n"
    "Set status to in_progress before starting, done when complete.\n"
    "Completing a task auto-unblocks dependents. Prefer tools over prose.\n\n"
    f"{SKILLS.get_descriptions()}"
)

TODO = TodoManager()
print(f"Tools ({len(TOOLS)}): {[t['function']['name'] for t in TOOLS]}")

### Step 3: Agent Loop

Same loop with all previous features (compaction, nag, skill loading). Task management works through the dispatch map.

In [ ]:
def agent_loop(messages: list):
    '''Agent loop with persistent task system.'''
    turn = 0
    rounds_since_todo = 0
    compact_requested = False
    while True:
        turn += 1

        # Compaction layers
        messages[:] = micro_compact(messages)
        est = estimate_tokens(messages)
        if est > TOKEN_THRESHOLD or compact_requested:
            print(f"  [compact] Auto-compacting...")
            messages[:] = auto_compact(messages)
            compact_requested = False

        all_messages = [{"role": "system", "content": SYSTEM}] + messages

        if rounds_since_todo >= 3:
            all_messages.append({"role": "user", "content": "<reminder>Update your todos/tasks.</reminder>"})
            print(f"  [nag] Injected task reminder")

        print(f"\n{'='*60}")
        print(f"Turn {turn} - LLM Input ({len(all_messages)} msgs, ~{estimate_tokens(all_messages)} tokens):")
        print(f"{'='*60}")
        for j, m in enumerate(all_messages):
            role = m["role"]
            if role == "system":
                print(f"  [{j}] system: {m['content'][:80]}...")
            elif role == "user":
                content = m.get("content", "")
                if isinstance(content, str):
                    print(f"  [{j}] user: {content[:100]}")
                else:
                    print(f"  [{j}] user: (tool results)")
            elif role == "assistant":
                tc = m.get("tool_calls")
                if tc:
                    print(f"  [{j}] assistant: (called {len(tc)} tool(s))")
                else:
                    print(f"  [{j}] assistant: {str(m.get('content', ''))[:100]}")
            elif role == "tool":
                print(f"  [{j}] tool: {m['content'][:80]}")

        response = client.chat.completions.create(
            model=MODEL,
            messages=all_messages,
            tools=TOOLS,
        )
        msg = response.choices[0].message

        print(f"\nTurn {turn} - LLM Output:")
        print(f"  content:    {msg.content}")
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  tool_call:  {tc.function.name}({tc.function.arguments[:100]})")
        else:
            print(f"  tool_calls: None (done!)")

        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)

        if not msg.tool_calls:
            print(f"\n>>> Agent finished in {turn} turn(s)")
            print(f"\n--- Task Board ---")
            print(TASKS.list_all())
            return

        used_todo = False
        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            handler = TOOL_HANDLERS.get(name)
            try:
                output = handler(**args) if handler else f"Unknown tool: {name}"
            except Exception as e:
                output = f"Error: {e}"
            print(f"\n  [exec] {name}: {json.dumps(args)[:100]}")
            print(f"  [result] {str(output)[:300]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(output),
            })
            if name in ("todo", "task_create", "task_update", "task_list"):
                used_todo = True
            if name == "compact":
                compact_requested = True

        rounds_since_todo = 0 if used_todo else rounds_since_todo + 1

## Try It!

Try a project that needs dependencies:
1. `Build a Python calculator package: create the module, add tests, then run them`
2. `List tasks` (see the task board)

Watch for:
- `blockedBy` arrays being set on dependent tasks
- Auto-unblocking when a task completes
- Tasks surviving in `.tasks/` directory (check with `ls .tasks/`)

In [ ]:
# Interactive REPL
TASKS = TaskManager()
TODO = TodoManager()
history = []
while True:
    try:
        query = input("s07 >> ")
    except (EOFError, KeyboardInterrupt):
        break
    if query.strip().lower() in ("q", "exit", ""):
        break
    history.append({"role": "user", "content": query})
    agent_loop(history)
    print()

## What Changed From S06

| Component      | Before (S06)           | After (S07)                              |
|----------------|------------------------|------------------------------------------|
| Tools          | 8                      | 12 (+task_create/update/list/get)        |
| Planning       | In-memory TodoManager  | + file-based TaskManager                 |
| Dependencies   | None                   | `blockedBy` arrays, auto-unblocking      |
| Persistence    | Lost on restart        | JSON files in `.tasks/` survive restarts |

> **Key insight**: File-based tasks persist across sessions. Dependencies create a DAG — the model can only work on unblocked tasks, enforcing correct ordering.

---

**Next: Session 08 — Background Tasks** → run slow operations in the background; the agent keeps thinking.

# S08: Background Tasks

`s01 > s02 > s03 > s04 > s05 > s06 | s07 > [ s08 ] s09 > s10 > s11 > s12`

> *"Run slow operations in the background; the agent keeps thinking"*
>
> **Harness layer**: Background execution — daemon threads with a notification queue.

## Problem

Some commands take minutes: `pip install`, `npm run build`, `pytest` on a large suite. The agent blocks on each one, wasting time it could spend reading files or planning the next step. We need **non-blocking execution** — fire off a command, keep working, check back later.

## Solution

```
Agent Loop
  |
  +-- background_run("pytest")
  |     +-> spawns daemon thread, returns task_id immediately
  |
  +-- (agent keeps working on other tools...)
  |
  +-- drain_notifications()   <-- before each LLM call
  |     +-> collects completed results, injects into messages
  |
  +-- background_check(task_id)   <-- explicit poll
```

The notification queue drains **before** each LLM call, so the model always sees fresh results.

### Step 1: BackgroundManager Class

Manages daemon threads for background command execution. `run()` spawns a thread and returns immediately. `check()` polls status. `drain_notifications()` collects all completed results.

In [ ]:
import threading
import queue

class BackgroundManager:
    '''Manages background command execution with daemon threads.'''

    def __init__(self):
        self._tasks = {}  # task_id -> {status, output, command}
        self._notifications = queue.Queue()
        self._next_id = 1
        self._lock = threading.Lock()

    def run(self, command: str) -> str:
        '''Spawn a daemon thread to run command. Returns immediately.'''
        with self._lock:
            task_id = f"bg_{self._next_id}"
            self._next_id += 1
        self._tasks[task_id] = {"status": "running", "output": None, "command": command}
        thread = threading.Thread(target=self._execute, args=(task_id, command), daemon=True)
        thread.start()
        return f"Started background task {task_id}: {command[:80]}"

    def _execute(self, task_id: str, command: str):
        '''Run command in thread, push result to notification queue.'''
        try:
            r = subprocess.run(
                ["powershell", "-NoProfile", "-Command", command],
                cwd=str(WORKDIR), capture_output=True, text=True, timeout=300,
            )
            output = (r.stdout + r.stderr).strip()[:50000]
        except subprocess.TimeoutExpired:
            output = "Error: Timeout (300s)"
        except Exception as e:
            output = f"Error: {e}"
        self._tasks[task_id]["status"] = "done"
        self._tasks[task_id]["output"] = output
        self._notifications.put({"task_id": task_id, "command": command, "output": output})

    def check(self, task_id: str) -> str:
        '''Poll status of a background task.'''
        task = self._tasks.get(task_id)
        if not task:
            return f"Unknown task: {task_id}"
        if task["status"] == "running":
            return f"{task_id} is still running: {task['command'][:60]}"
        return f"{task_id} completed:\n{task['output'][:5000]}"

    def drain_notifications(self) -> list:
        '''Collect all completed background task results.'''
        results = []
        while not self._notifications.empty():
            try:
                results.append(self._notifications.get_nowait())
            except queue.Empty:
                break
        return results


BG = BackgroundManager()
print("BackgroundManager ready")

### Step 2: Background Tools & Updated Dispatch

Two new tools: `background_run` (fire and forget) and `background_check` (poll status). The notification drain happens in the agent loop.

In [ ]:
BG_TOOLS = [
    {"type": "function", "function": {
        "name": "background_run",
        "description": "Run a PowerShell command in the background. Returns immediately with a task_id.",
        "parameters": {
            "type": "object",
            "properties": {"command": {"type": "string"}},
            "required": ["command"],
        },
    }},
    {"type": "function", "function": {
        "name": "background_check",
        "description": "Check status of a background task by task_id.",
        "parameters": {
            "type": "object",
            "properties": {"task_id": {"type": "string"}},
            "required": ["task_id"],
        },
    }},
]

TOOLS = CHILD_TOOLS + [
    TASK_TOOL,
    {"type": "function", "function": {
        "name": "load_skill",
        "description": "Load full content of a skill by name.",
        "parameters": {"type": "object", "properties": {"name": {"type": "string"}}, "required": ["name"]},
    }},
    {"type": "function", "function": {
        "name": "compact",
        "description": "Compress conversation context.",
        "parameters": {"type": "object", "properties": {}},
    }},
] + TASK_MGMT_TOOLS + BG_TOOLS

TOOL_HANDLERS = {
    **CHILD_HANDLERS,
    "task":             lambda **kw: run_subagent(kw["prompt"]),
    "load_skill":       lambda **kw: SKILLS.get_content(kw["name"]),
    "compact":          run_compact,
    "task_create":      lambda **kw: TASKS.create(kw["title"], kw.get("description", ""), kw.get("blocked_by")),
    "task_update":      lambda **kw: TASKS.update(kw["task_id"], kw["status"]),
    "task_list":        lambda **kw: TASKS.list_all(),
    "task_get":         lambda **kw: TASKS.get(kw["task_id"]),
    "background_run":   lambda **kw: BG.run(kw["command"]),
    "background_check": lambda **kw: BG.check(kw["task_id"]),
}

SYSTEM = (
    f"You are a coding agent at {WORKDIR}.\n"
    "Use task_create/task_update for persistent task tracking.\n"
    "Use background_run for slow commands (builds, tests, installs). "
    "Use background_check to poll results. "
    "Notifications appear automatically before each turn.\n"
    "Prefer tools over prose.\n\n"
    f"{SKILLS.get_descriptions()}"
)

TODO = TodoManager()
TASKS = TaskManager()
BG = BackgroundManager()
print(f"Tools ({len(TOOLS)}): {[t['function']['name'] for t in TOOLS]}")

### Step 3: Agent Loop with Notification Drain

The key addition: `drain_notifications()` runs before each LLM call. Completed background results are injected as user messages so the model sees them.

In [ ]:
def agent_loop(messages: list):
    '''Agent loop with background task notification drain.'''
    turn = 0
    rounds_since_todo = 0
    compact_requested = False
    while True:
        turn += 1

        # Compaction layers
        messages[:] = micro_compact(messages)
        est = estimate_tokens(messages)
        if est > TOKEN_THRESHOLD or compact_requested:
            print(f"  [compact] Auto-compacting...")
            messages[:] = auto_compact(messages)
            compact_requested = False

        # Drain background notifications
        notifications = BG.drain_notifications()
        for note in notifications:
            notify_msg = (
                f"[Background completed] {note['task_id']}: {note['command'][:60]}\n"
                f"Output: {note['output'][:2000]}"
            )
            messages.append({"role": "user", "content": notify_msg})
            print(f"  [bg-notify] {note['task_id']} completed")

        all_messages = [{"role": "system", "content": SYSTEM}] + messages

        if rounds_since_todo >= 3:
            all_messages.append({"role": "user", "content": "<reminder>Update your tasks.</reminder>"})
            print(f"  [nag] Injected task reminder")

        print(f"\n{'='*60}")
        print(f"Turn {turn} - LLM Input ({len(all_messages)} msgs, ~{estimate_tokens(all_messages)} tokens):")
        print(f"{'='*60}")
        for j, m in enumerate(all_messages):
            role = m["role"]
            if role == "system":
                print(f"  [{j}] system: {m['content'][:80]}...")
            elif role == "user":
                content = m.get("content", "")
                if isinstance(content, str):
                    print(f"  [{j}] user: {content[:100]}")
                else:
                    print(f"  [{j}] user: (tool results)")
            elif role == "assistant":
                tc = m.get("tool_calls")
                if tc:
                    print(f"  [{j}] assistant: (called {len(tc)} tool(s))")
                else:
                    print(f"  [{j}] assistant: {str(m.get('content', ''))[:100]}")
            elif role == "tool":
                print(f"  [{j}] tool: {m['content'][:80]}")

        response = client.chat.completions.create(
            model=MODEL,
            messages=all_messages,
            tools=TOOLS,
        )
        msg = response.choices[0].message

        print(f"\nTurn {turn} - LLM Output:")
        print(f"  content:    {msg.content}")
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  tool_call:  {tc.function.name}({tc.function.arguments[:100]})")
        else:
            print(f"  tool_calls: None (done!)")

        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)

        if not msg.tool_calls:
            print(f"\n>>> Agent finished in {turn} turn(s)")
            return

        used_todo = False
        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            handler = TOOL_HANDLERS.get(name)
            try:
                output = handler(**args) if handler else f"Unknown tool: {name}"
            except Exception as e:
                output = f"Error: {e}"
            print(f"\n  [exec] {name}: {json.dumps(args)[:100]}")
            print(f"  [result] {str(output)[:300]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(output),
            })
            if name in ("todo", "task_create", "task_update", "task_list"):
                used_todo = True
            if name == "compact":
                compact_requested = True

        rounds_since_todo = 0 if used_todo else rounds_since_todo + 1

## Try It!

Try running something slow in the background:
1. `Run 'Start-Sleep 5; Write-Output done' in the background, then read pyproject.toml while it runs`
2. `Run pytest in the background and keep working on other files`

Watch for:
- `background_run` returning immediately with a task_id
- The agent continuing with other work
- `[bg-notify]` messages appearing when the background task completes

In [ ]:
# Interactive REPL
TODO = TodoManager()
TASKS = TaskManager()
BG = BackgroundManager()
history = []
while True:
    try:
        query = input("s08 >> ")
    except (EOFError, KeyboardInterrupt):
        break
    if query.strip().lower() in ("q", "exit", ""):
        break
    history.append({"role": "user", "content": query})
    agent_loop(history)
    print()

## What Changed From S07

| Component       | Before (S07)           | After (S08)                                |
|-----------------|------------------------|--------------------------------------------|
| Tools           | 12                     | 14 (+background_run, background_check)     |
| Execution       | Synchronous only       | + daemon threads for background commands   |
| Notifications   | None                   | Queue drained before each LLM call         |
| Concurrency     | Blocked on each tool   | Fire-and-forget + poll pattern             |

> **Key insight**: The agent doesn't have to wait. Background threads run commands while the model keeps thinking. The notification queue ensures results reach the model at the right time.

---

**Next: Session 09** → continuing the build...